[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/21_generative_model_rl.ipynb)

# 21. RL for diffusion and flow generation — trajectory-level policy optimization

이전 버전은 DDPO를 Gaussian transition 하나의 ratio로만 보여줬고, deterministic flow velocity에 reward scale을 곱한 것을 RL처럼 보이게 만들었다.

이번 버전은 **denoising 전체 trajectory를 MDP로 보고 각 reverse transition의 log-probability를 저장/재평가하는 DDPO 구조**를 구현한다. deterministic ODE flow에는 transition density가 없으므로 같은 PPO ratio를 그대로 쓸 수 없다는 점도 구분한다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Reverse diffusion transition as an RL policy action

DDPO에서는 state를 `(x_t,t,condition)`으로 보고 action을 다음 latent `x_{t-1}`로 볼 수 있다. reverse process가 Gaussian이면 policy는 `p_theta(x_{t-1}|x_t,c)=N(mu_theta, sigma_t^2 I)`가 되고, 각 denoising step의 log-probability를 계산할 수 있다.


In [ ]:
class TinyDenoiser(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x_t, t):
        model_input = torch.cat(
            [x_t, t[:, None]],
            dim=-1,
        )
        return self.net(model_input)


def gaussian_log_probability(sample, mean, sigma):
    variance = sigma.square()
    log_normalizer = torch.log(
        2 * math.pi * variance
    )

    elementwise = -0.5 * (
        (sample - mean).square() / variance
        + log_normalizer
    )
    return elementwise.sum(dim=-1)


def transition_mean(model, x_t, t, step_size):
    predicted_direction = model(x_t, t)
    return x_t + step_size * predicted_direction


old_policy = TinyDenoiser().to(device)
current_policy = TinyDenoiser().to(device)
current_policy.load_state_dict(old_policy.state_dict())


## 2. Roll out a complete denoising trajectory with the old policy

on-policy rollout에서 시작 noise `x_T`부터 여러 stochastic reverse transition을 sampling한다. 각 step에서 state, sampled next state, old log-prob을 저장해야 나중에 PPO-style update를 할 수 있다.


In [ ]:
batch_size = 6
num_steps = 5
step_size = -0.15
transition_sigma = torch.tensor(0.12, device=device)

x_t = torch.randn(batch_size, 2, device=device)

states = []
next_states = []
times = []
old_log_probs = []

with torch.no_grad():
    for step in range(num_steps):
        t = torch.full(
            (batch_size,),
            1.0 - step / num_steps,
            device=device,
        )

        mean = transition_mean(
            old_policy,
            x_t,
            t,
            step_size,
        )
        noise = torch.randn_like(x_t)
        x_next = mean + transition_sigma * noise

        log_prob = gaussian_log_probability(
            x_next,
            mean,
            transition_sigma,
        )

        states.append(x_t.clone())
        next_states.append(x_next.clone())
        times.append(t.clone())
        old_log_probs.append(log_prob.clone())

        x_t = x_next

final_samples = x_t

states = torch.stack(states, dim=1)
next_states = torch.stack(next_states, dim=1)
times = torch.stack(times, dim=1)
old_log_probs = torch.stack(old_log_probs, dim=1)

print("states:", states.shape)
print("transition log-probs:", old_log_probs.shape)


## 3. Terminal reward becomes a trajectory advantage

image-generation RL처럼 reward가 final sample에만 있다면 같은 trajectory-level advantage가 그 trajectory의 모든 denoising transitions에 credit assignment된다. 실제 DDPO에서는 prompt별 reward normalization 등도 사용할 수 있다.


In [ ]:
target = torch.tensor(
    [0.0, 0.0],
    device=device,
)
reward = -(final_samples - target).square().sum(dim=-1)
advantage = (
    reward - reward.mean()
) / (reward.std(unbiased=False) + 1e-6)

print("reward:", reward)
print("advantage:", advantage)


## 4. Re-evaluate every sampled transition under the current policy

PPO/DDPO update 시 trajectory를 새로 sampling하는 대신 old policy가 만든 `(x_t,x_{t-1})` pair를 current policy의 Gaussian transition 아래에서 다시 평가해 importance ratio를 계산한다.


In [ ]:
new_log_probs = []

for step in range(num_steps):
    state_t = states[:, step]
    sampled_next = next_states[:, step]
    time_t = times[:, step]

    new_mean = transition_mean(
        current_policy,
        state_t,
        time_t,
        step_size,
    )
    new_log_prob = gaussian_log_probability(
        sampled_next,
        new_mean,
        transition_sigma,
    )
    new_log_probs.append(new_log_prob)

new_log_probs = torch.stack(new_log_probs, dim=1)

print("new transition log-probs:", new_log_probs.shape)


## 5. DDPO-style PPO clipped objective across denoising steps

각 reverse transition의 ratio에 terminal trajectory advantage를 broadcast하고 PPO clipping을 적용한다. 이것이 single Gaussian ratio toy와 실제 multi-step denoising policy optimization의 가장 큰 차이다.


In [ ]:
ratio = torch.exp(
    new_log_probs - old_log_probs.detach()
)
trajectory_advantage = advantage[:, None]

unclipped = ratio * trajectory_advantage
clipped = ratio.clamp(0.8, 1.2) * trajectory_advantage

ddpo_loss = -torch.minimum(
    unclipped,
    clipped,
).mean()

print("DDPO loss:", ddpo_loss.item())
print("ratio shape:", ratio.shape)


## 6. Reward-weighted denoising loss is a different method

reward로 diffusion pretraining loss를 재가중하는 방법은 유용할 수 있지만, transition log-prob policy gradient를 사용하는 DDPO와 같은 objective는 아니다. 둘을 구분해서 비교한다.


In [ ]:
predicted_noise = torch.randn(
    batch_size, 2,
    device=device,
    requires_grad=True,
)
noise_target = torch.randn_like(predicted_noise)

reward_weight = torch.softmax(reward.detach(), dim=0)
per_sample_mse = F.mse_loss(
    predicted_noise,
    noise_target,
    reduction="none",
).mean(dim=-1)

reward_weighted_loss = (
    reward_weight * per_sample_mse
).sum()

print("reward-weighted denoising loss:", reward_weighted_loss.item())


## 7. Deterministic flow ODE does not automatically have a PPO transition ratio

deterministic flow sampler `x_{t+dt}=x_t+dt*v_theta(x_t,t)`은 conditional Gaussian transition distribution을 정의하지 않는다. 따라서 아래처럼 velocity에 reward scalar를 곱하는 것은 guidance heuristic일 수는 있어도 DDPO/GRPO policy gradient가 아니다. flow-model RL에서 PPO식 ratio를 쓰려면 stochastic policy/SDE transition을 정의하거나, 별도의 flow-RL derivation을 사용해야 한다.


In [ ]:
x_t = torch.tensor([[1.0, -1.0]], device=device)
velocity = torch.tensor([[-0.4, 0.3]], device=device)
dt = 0.2

deterministic_next = x_t + dt * velocity

print("deterministic flow step:", deterministic_next)
print("no transition log-probability exists in this deterministic update")


## References and provenance

**DDPO** — Black et al., *Training Diffusion Models with Reinforcement Learning*. denoising process를 multi-step MDP로 보고 reverse transition distribution을 policy로 최적화하는 핵심을 반영했다.

**Reward-weighted diffusion matching** — reward-weighted regression 계열은 DDPO와 별도 objective로 구분했다.

**Flow-model RL** — deterministic ODE에 Gaussian DDPO ratio를 임의로 붙이지 않는다. 구체 Flow-GRPO/flow-policy RL을 구현하려면 해당 논문의 stochastic construction 또는 specialized objective를 그대로 가져와야 한다. 이 노트북에서는 검증되지 않은 shortcut을 제거했다.
